# W&B Confusion Matrix Provenance Notebook

- Original source: Derrick Chan-Sew
- Purpose: reconstruct the supplemental ViT confusion matrix (Fig S3) from the downloaded W&B table artifact
- Checkpoint: `smqmqi14` (VT-Reduced Reflection, W&B training run `smqmqi14`, confusion matrix artifact from run `76zuzb4h`)
- Artifact: `confusion_matrix_smqmqi14.table.json` — bundled at `assets/figure_data/`

**Dependency:** `pip install seaborn` (not in base `environment-train-eval.yml`)

This notebook is retained as a provenance artifact. It does not require a GPU or large HDF5 datasets — it renders directly from the stored W&B table JSON.

In [ ]:
from pathlib import Path
import os

cwd = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / 'configs').exists()), None
)
if REPO_ROOT is None:
    raise FileNotFoundError(f'Could not locate repo root from {cwd}')

ARTIFACT_JSON = REPO_ROOT / 'assets' / 'figure_data' / 'confusion_matrix_smqmqi14.table.json'
OUTPUT_PNG            = Path('confusion_matrix_smqmqi14.png')
OUTPUT_PNG_NORMALIZED = Path('confusion_matrix_smqmqi14_normalized.png')
NUM_CLASSES = 99

assert ARTIFACT_JSON.exists(), f'Artifact not found: {ARTIFACT_JSON}'
print(f'Artifact: {ARTIFACT_JSON}')

In [ ]:
import json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

with ARTIFACT_JSON.open("r") as f:
    data = json.load(f)

df = pd.DataFrame(data["data"], columns=data["columns"])
conf_matrix = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=float)
for _, row in df.iterrows():
    actual = int(row["Actual"])
    predicted = int(row["Predicted"])
    weight = float(row["nPredictions"])
    conf_matrix[actual, predicted] += weight

plt.figure(figsize=(16, 12))
sns.heatmap(conf_matrix, cmap="Blues", annot=False)
plt.title("ViT Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.savefig(OUTPUT_PNG, bbox_inches="tight")
plt.show()


In [ ]:
row_sums = conf_matrix.sum(axis=1, keepdims=True)
conf_matrix_norm = np.divide(
    conf_matrix,
    row_sums,
    out=np.zeros_like(conf_matrix),
    where=row_sums != 0,
)

plt.figure(figsize=(20, 16))
sns.heatmap(conf_matrix_norm, cmap="viridis", annot=False, cbar=True)
plt.title("ViT Confusion Matrix (Row-normalized)")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.savefig(OUTPUT_PNG_NORMALIZED, bbox_inches="tight")
plt.show()
